# Annihilate free Colab smoke test

This notebook is tuned for the free Google Colab GPU tier. It uses `HuggingFaceTB/SmolLM2-1.7B-Instruct` with very small prompt slices and a tiny number of trials so you can verify that Annihilate installs, sees the GPU, loads the model, and starts optimization.

Before running: choose **Runtime > Change runtime type > T4 GPU** or another available GPU. Free Colab availability is not guaranteed.

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

In [ ]:
!python -m pip install -U pip
!python -m pip install -U annihilate-llm

The next cell writes a deliberately small config for free Colab. It is for a smoke test, not a quality run. Increase prompt counts and `n_trials` only if you have enough GPU time and memory.

In [ ]:
%%writefile config.toml
model = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
dtypes = ["float16"]
quantization = "none"
device_map = "auto"
max_memory = { "0" = "14GB", "cpu" = "12GB" }
offload_outputs_to_cpu = true
batch_size = 1
max_batch_size = 2
max_response_length = 48
print_responses = false
print_residual_geometry = false
plot_residuals = false
kl_divergence_target = 0.01
row_normalization = "full"
full_normalization_lora_rank = 2
n_trials = 3
n_startup_trials = 3
seed = 42
study_checkpoint_dir = "checkpoints_colab"

[good_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "train[:32]"
column = "text"
residual_plot_label = '"Harmless" prompts'
residual_plot_color = "royalblue"

[bad_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "train[:32]"
column = "text"
residual_plot_label = '"Harmful" prompts'
residual_plot_color = "darkorange"

[good_evaluation_prompts]
dataset = "mlabonne/harmless_alpaca"
split = "test[:16]"
column = "text"

[bad_evaluation_prompts]
dataset = "mlabonne/harmful_behaviors"
split = "test[:16]"
column = "text"


In [ ]:
!cat config.toml

Run the smoke test. When optimization finishes, choose **Exit program** unless you want to manually save, chat, benchmark, or upload from the interactive menu.

In [ ]:
!annihilate